# Autograd Test

In [ ]:
import time
import autograd.numpy as anp
from autograd import grad

# Tiny vocabulary
words = ["<pad>", "<eos>", "the", "cat", "sat", "on", "mat", "dog", "ran"]
V = len(words)
w2i = {w: i for i, w in enumerate(words)}

def encode(sentence):
    return [w2i[w] for w in sentence.split()] + [w2i["<eos>"]]

# Word-reversal pairs
pairs = [
    ("the cat sat",      "sat cat the"),
    ("the dog ran",      "ran dog the"),
    ("cat sat on mat",   "mat on sat cat"),
]

D = 32  # hidden dim

anp.random.seed(42)
params = {
    "E":    anp.random.randn(V, D) * 0.1,   # embedding matrix
    "Wenc": anp.random.randn(D, D) * 0.1,   # encoder projection
    "Wdec": anp.random.randn(D, D) * 0.1,   # decoder query projection
    "Wout": anp.random.randn(D, V) * 0.1,   # output projection
}

def softmax(x):
    e = anp.exp(x - anp.max(x))
    return e / anp.sum(e)

def forward(params, src_ids, tgt_ids):
    enc = anp.tanh(params["E"][src_ids] @ params["Wenc"])   # (S, D)
    loss = 0.0
    for t, tgt_id in enumerate(tgt_ids):
        query  = enc[min(t, len(enc) - 1)] @ params["Wdec"] # (D,)
        attn   = softmax(enc @ query)                         # (S,)
        ctx    = attn @ enc                                   # (D,)
        probs  = softmax(ctx @ params["Wout"])                # (V,)
        loss  += -anp.log(probs[tgt_id] + 1e-9)
    return loss / len(tgt_ids)

grad_fn = grad(forward)

# Training loop
lr = 0.2
t0 = time.time()

for epoch in range(200):
    total_loss = 0.0
    for src, tgt in pairs:
        src_ids, tgt_ids = encode(src), encode(tgt)
        grads = grad_fn(params, src_ids, tgt_ids)
        for k in params:
            params[k] = params[k] - lr * grads[k]
        total_loss += forward(params, src_ids, tgt_ids)
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss / len(pairs):.4f}")

elapsed = time.time() - t0
print(f"\n{elapsed:.1f}s total  |  {elapsed/10:.2f}s per epoch")

# Quick inference check
def predict(params, sentence):
    src_ids = encode(sentence)
    enc = anp.tanh(params["E"][src_ids] @ params["Wenc"])
    output = []
    for t in range(len(src_ids) + 2):
        query  = enc[min(t, len(enc) - 1)] @ params["Wdec"]
        ctx    = softmax(enc @ query) @ enc
        tok    = int(anp.argmax(softmax(ctx @ params["Wout"])))
        if tok == w2i["<eos>"]: break
        output.append(words[tok])
    return " ".join(output)

print()
for src, expected in pairs:
    print(f"  '{src}' → '{predict(params, src)}'  (expected: '{expected.replace('<eos>', 
'').strip()}')")